In [1]:
# 1. Actualizar el gestor de paquetes pip
!python -m pip install --upgrade pip

# 2. Asegurar que spaCy esté instalado correctamente
!python -m pip install spacy

In [2]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     ------------- -------------------------- 4.5/12.8 MB 17.9 MB/s eta 0:00:01
     --------------------------------------  12.6/12.8 MB 27.2 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 25.9 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [3]:
# Instalamos 'click' y aseguramos otra librería clave que usa spaCy llamada 'typer'
!pip install click typer

In [5]:
import spacy

# Intentamos cargar el modelo que acabamos de instalar
nlp = spacy.load("en_core_web_sm")

In [6]:
# Le enseñamos a Jupyter la ubicación moderna de 'display'
from IPython.display import display

In [7]:
import spacy
from spacy import displacy
from IPython.display import display, HTML

In [8]:
!pip install pandas requests spacy notebook

In [ ]:
# Celda 1: Importación de librerías y configuración de spaCy
import requests
import pandas as pd
import time

# Cargamos el modelo en inglés de spaCy
nlp = spacy.load("en_core_web_sm")

# Categorías clave solicitadas para el negocio
CATEGORIAS = ["backend", "frontend", "cloud", "datascience"]

print("Entorno inicializado correctamente.")

Entorno inicializado correctamente.


In [10]:
# Celda 2 (Expandida): Extracción, Filtrado y Reconocimiento de Entidades (NER)
def extraer_y_filtrar_articulos(tag_busqueda, max_articulos=1250):
    """
    Realiza scraping de la API de dev.to, procesa con spaCy para filtrar calidad
    y extrae Entidades Nombradas (NER) como palabras clave técnicas.
    """
    url = "https://dev.to/api/articles"
    articulos_procesados = []
    page = 1
    
    print(f"🚀 Iniciando extracción y análisis NER para la categoría: {tag_busqueda}...")
    
    while len(articulos_procesados) < max_articulos:
        params = {
            "tag": tag_busqueda,
            "page": page,
            "per_page": 50
        }
        
        try:
            response = requests.get(url, params=params, timeout=10)
            if response.status_code != 200:
                print(f"⚠️ Error en página {page}. Código: {response.status_code}")
                break
                
            data = response.json()
            if not data:
                break  # No hay más artículos
                
            for item in data:
                if len(articulos_procesados) >= max_articulos:
                    break
                    
                titulo = item.get("title", "")
                texto_cuerpo = item.get("description", "")
                
                if not texto_cuerpo or len(texto_cuerpo) < 20:
                    continue
                    
                # Procesamiento con el modelo de spaCy
                doc = nlp(texto_cuerpo)
                
                # 1. Validación de calidad (Filtro por número de tokens significativos)
                tokens_validos = [token.text for token in doc if not token.is_stop and not token.is_punct]
                if len(tokens_validos) <= 5:
                    continue
                
                # 2. EXTRACCIÓN DE ENTIDADES (NER): Extraemos palabras clave (ej: organizaciones, productos, misceláneos)
                # Limpiamos duplicados, pasamos a minúsculas y filtramos palabras vacías remanentes
                entidades_tecnicas = set()
                for ent in doc.ents:
                    ent_texto = ent.text.strip()
                    # Filtro básico de longitud y ruido
                    if len(ent_texto) > 1 and not ent.root.is_stop:
                        entidades_tecnicas.add(ent_texto)
                
                # Agregamos el registro enriquecido al dataset
                articulos_procesados.append({
                    "titulo": titulo,
                    "texto": texto_cuerpo,
                    "categoria": tag_busqueda,
                    # Convertimos el set a una lista separada por comas o formato JSON-friendly
                    "palabras_clave": list(entidades_tecnicas) 
                })
            
            page += 1
            time.sleep(1)  # Rate limiting preventivo
            
        except Exception as e:
            print(f"❌ Error en la petición: {e}")
            break
            
    print(f"✅ Total recolectado y enriquecido para '{tag_busqueda}': {len(articulos_procesados)} artículos.")
    return articulos_procesados

In [11]:
# Celda 3: Ejecución del pipeline y consolidación del Dataset
dataset_completo = []

# Iteramos sobre cada categoría de negocio
for cat in CATEGORIAS:
    articulos_cat = extraer_y_filtrar_articulos(tag_busqueda=cat, max_articulos=1250)
    dataset_completo.extend(articulos_cat)

# Convertimos a DataFrame de Pandas
df = pd.DataFrame(dataset_completo)

# Mezclamos los datos (shuffle) para evitar sesgos por orden en el entrenamiento posterior
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n🎉 ¡Dataset creado con éxito! Total de registros: {len(df)}")

🚀 Iniciando extracción y análisis NER para la categoría: backend...
✅ Total recolectado y enriquecido para 'backend': 1250 artículos.
🚀 Iniciando extracción y análisis NER para la categoría: frontend...
✅ Total recolectado y enriquecido para 'frontend': 1250 artículos.
🚀 Iniciando extracción y análisis NER para la categoría: cloud...
✅ Total recolectado y enriquecido para 'cloud': 1250 artículos.
🚀 Iniciando extracción y análisis NER para la categoría: datascience...
✅ Total recolectado y enriquecido para 'datascience': 1250 artículos.

🎉 ¡Dataset creado con éxito! Total de registros: 5000


In [12]:
# Celda 4: Visualización rápida y exportación a CSV
# Mostrar distribución de clases
print("\n--- Distribución por Categorías ---")
print(df["categoria"].value_counts())

# Guardar en la carpeta local para la posterior carga en OCI
df.to_csv("dataset_tecnico_hackathon_prueba_4.csv", index=False, encoding="utf-8")
print("\n Archivo 'dataset_tecnico_hackathon_prueba_4.csv' guardado exitosamente.")


--- Distribución por Categorías ---
categoria
frontend       1250
cloud          1250
backend        1250
datascience    1250
Name: count, dtype: int64

 Archivo 'dataset_tecnico_hackathon_prueba_4.csv' guardado exitosamente.
